# Day 5: Session 5A - The Split-Apply-Combine Pattern

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/5a_grouping_data.html)

Date: 09/04/2026

In [2]:
import pandas as pd
url = 'https://eds-217-essential-python.github.io/data/messy_field_survey.csv'
survey = pd.read_csv(url)

survey = survey.drop_duplicates()

survey["site"] = survey["site"].str.strip().str.lower().str.replace("-", "_")
# survey['site'] = survey['site'].str.lower()
# survey['site'] = survey['site'].str.replace('-', '_')

survey["pH"] = survey["pH"].str.replace(",", ".").astype(float)
# survey['pH'] = survey['pH'].astype(float)

survey = survey.dropna(
    subset=["temperature_c", "dissolved_oxygen_mg_L", "conductivity_uS_cm"]
)
survey["n_replicates"] = survey["n_replicates"].fillna(1).astype(int)
#survey["n_replicates"] = survey["n_replicates"].astype(int)

survey = survey[survey["temperature_c"] > -100].copy()
survey = survey.rename(columns={"collection date": "collection_date"})

survey.shape


(255, 7)

In [3]:
survey.head()

,site,collection_date,temperature_c,pH,dissolved_oxygen_mg_L,conductivity_uS_cm,n_replicates
0,site_f,2025-07-28,24.1,5.89,5.51,899.1,4
1,site_d,2025-08-05,12.5,7.74,9.97,300.1,4
2,site_c,2025-07-04,22.6,6.42,7.19,815.8,4
3,site_d,2025-08-15,16.0,7.74,9.73,250.7,3
4,site_a,2025-06-24,14.7,7.64,9.19,338.8,3


In [6]:
survey['dissolved_oxygen_mg_L'].mean()

8.062039215686275

In [9]:
site_d = survey[survey['site'] == 'site_d']
print(site_d['dissolved_oxygen_mg_L'].mean())

site_f = survey[survey['site'] == 'site_f']
print(site_f['dissolved_oxygen_mg_L'].mean())

10.053333333333333
6.180232558139535


In [10]:
# always need to specify which column you want to group by
grouped = survey.groupby('site')

In [11]:
type(grouped)

pandas.core.groupby.generic.DataFrameGroupBy

In [14]:
# this is the mean of the dissolved oxygen at each site in one step. since we used the groupby function
result = grouped['dissolved_oxygen_mg_L'].mean()

In [ ]:
# it just made a series
print(result)
print(type(result))

site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64
<class 'pandas.core.series.Series'>


In [ ]:
# now i can find the specific result for a site by calling the column.
result['site_a']

9.485909090909091

In [19]:
result.idxmax()

'site_d'

In [20]:
result.idxmin()

'site_f'

In [ ]:
# step 1 - make the groupby object
grouped = survey.groupby('site')

# step 2 - aggregate on the object
grouped['dissolved_oxygen_mg_L'].mean()

site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64

In [24]:
# this is the best way to do it. the most efficient. 

survey.groupby('site')['dissolved_oxygen_mg_L'].mean()

site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64

In [26]:
# this is the format for this

# df.groupby('key')['column'].aggregation()
#          split      pick        apply

In [27]:
# groupby('key') says which piles to make. The key is the column whose values name the groups.
# ['column'] says which column to do analysis on.
# .mean() says what arithmetic.

In [28]:
mean_do = survey.groupby('site')['dissolved_oxygen_mg_L'].mean()

print(type(mean_do))
print(mean_do.index)

<class 'pandas.core.series.Series'>
Index(['site_a', 'site_b', 'site_c', 'site_d', 'site_e', 'site_f'], dtype='object', name='site')


In [29]:
mean_do['site_c']

6.873111111111111

In [37]:
mean_do.idxmax()

'site_d'

In [38]:
mean_do.idxmin()

'site_f'

In [34]:
result = survey.groupby('site')['temperature_c'].mean()
result.idxmax()

'site_f'

In [39]:
survey.groupby('site')['dissolved_oxygen_mg_L'].max()

site
site_a    11.12
site_b     9.86
site_c     7.99
site_d    11.43
site_e     8.79
site_f     7.77
Name: dissolved_oxygen_mg_L, dtype: float64

In [40]:
survey.groupby('site')['dissolved_oxygen_mg_L'].min()

site
site_a    8.21
site_b    7.33
site_c    5.65
site_d    8.60
site_e    5.16
site_f    4.14
Name: dissolved_oxygen_mg_L, dtype: float64

In [44]:
grouped['n_replicates'].sum()

site
site_a    130
site_b    115
site_c    131
site_d    116
site_e    127
site_f    127
Name: n_replicates, dtype: int64

In [ ]:
# seeing how big the sample size is
survey.groupby('site')['dissolved_oxygen_mg_L'].count()

site
site_a    44
site_b    41
site_c    45
site_d    39
site_e    43
site_f    43
Name: dissolved_oxygen_mg_L, dtype: int64

In [48]:
# this is for the whole column

survey['dissolved_oxygen_mg_L'].count()

255

In [58]:
survey.groupby('site')['pH'].max()


site
site_a    7.92
site_b    7.41
site_c    7.10
site_d    8.33
site_e    7.47
site_f    6.98
Name: pH, dtype: float64

In [59]:
survey.groupby('site')['n_replicates'].sum()

site
site_a    130
site_b    115
site_c    131
site_d    116
site_e    127
site_f    127
Name: n_replicates, dtype: int64

What is the highest pH recorded at each site?
- site_a

How many bottles were filled in total at each site? (n_replicates counts bottles.)
- look above


In [60]:
def classify_ph(value):
    """Label a pH value as acidic, neutral, or alkaline."""
    if value < 6.5:
        return 'acidic'
    elif value > 7.5:
        return 'alkaline'
    else:
        return 'neutral'


survey['ph_class'] = survey['pH'].apply(classify_ph)
survey['ph_class'].value_counts()

ph_class
neutral     171
acidic       42
alkaline     42
Name: count, dtype: int64

In [61]:
survey.groupby('ph_class')['dissolved_oxygen_mg_L'].mean()

ph_class
acidic      6.350000
alkaline    9.948571
neutral     8.019181
Name: dissolved_oxygen_mg_L, dtype: float64

In [ ]:
# group by takes data and puts it bins.

In [62]:
print(survey.groupby('site')['dissolved_oxygen_mg_L'].count())
print(survey.groupby('site')['dissolved_oxygen_mg_L'].mean())
print(survey.groupby('site')['temperature_c'].mean())

site
site_a    44
site_b    41
site_c    45
site_d    39
site_e    43
site_f    43
Name: dissolved_oxygen_mg_L, dtype: int64
site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64
site
site_a    16.600000
site_b    18.102439
site_c    21.911111
site_d    15.179487
site_e    19.730233
site_f    23.297674
Name: temperature_c, dtype: float64
